# Group 32 — verified multi-edit pilot
Rohith: setup, implementation, and verification. GPU verification is pending until the final export succeeds. Select a GPU runtime (prefer A100). Run cells in order. This notebook performs required verification after the plan’s November 21 cutoff; outputs record actual completion dates.

Stages: hardware → setup → Drive → 1 group → 5 groups → resume check → 20 groups in both orders → verify/export. No manual dataset upload is required.

In [1]:
import sys, subprocess, shutil, json, os
from pathlib import Path
import torch
assert torch.cuda.is_available(), "Select a GPU runtime before proceeding."
print("Notebook Python:", sys.version.split()[0])
print("GPU:", torch.cuda.get_device_name(0))
print("GPU GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("Free disk GiB:", round(shutil.disk_usage('/content').free / 2**30, 1))
assert shutil.disk_usage('/content').free > 20 * 2**30, "Need at least 20 GiB free."
def run(command, cwd=None):
    with subprocess.Popen([str(x) for x in command], cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
            if process.wait():
                raise RuntimeError("Command failed; inspect output above. Do not continue.")
        except KeyboardInterrupt:
            process.terminate()
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill(); process.wait()
            raise


Notebook Python: 3.13.16
GPU: NVIDIA A100-SXM4-40GB
GPU GiB: 39.5
Free disk GiB: 235.6


## Install the pinned experiment environment
The notebook kernel may use Python 3.13; all model code runs in the separate Python 3.11 environment. The implementation checkout is pinned to a commit on first use and persisted in Drive below.

In [2]:
REPO_URL = "https://github.com/Gogireddy007/Multifact-Rome.git"
BRANCH = "rohith/final-multi-edit"
ROOT = Path('/content/group32_final_multi')
if not ROOT.exists():
    run(['git', 'clone', '--branch', BRANCH, REPO_URL, ROOT])
assert not subprocess.check_output(['git','-C',str(ROOT),'status','--porcelain','--untracked-files=no'], text=True).strip(), "Checkout has local edits."
SOURCE_COMMIT = subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'], text=True).strip()
print("Experiment source:", SOURCE_COMMIT)
run([sys.executable, ROOT/'scripts/setup.py'], cwd=ROOT)
PYTHON = ROOT/'.venv/bin/python'
run([PYTHON, '-m', 'unittest', 'discover', '-s', 'scripts', '-p', 'test_*.py', '-v'], cwd=ROOT)


Cloning into '/content/group32_final_multi'...
Experiment source: 44907cdcfa0fc0c0c4cc7026deca1c754cf9c2d5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 182.1 MB/s  0:00:00
Installed Python 3.11.17 in 872ms
 + cpython-3.11.17-linux-x86_64-gnu (python3.11)
Using CPython 3.11.17
Creating virtual environment with seed packages at: .venv
 + packaging==26.4
 + pip==26.3
 + setuptools==84.2.1
 + wheel==0.48.0
Activate with: source .venv/bin/activate
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.5/780.5 MB 34.4 MB/s  0:00:08
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 253.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 339.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 260.2 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 39.7 MB/s  0:00:07
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 255.2 MB/s  0:00:01
  

## Persistent output
Drive authorization may require your Google account confirmation. Keep the output name for resume; use a new name for intentional configuration/GPU changes. Stop if a previous run has a different commit; check out its recorded commit before resuming.

In [3]:
from google.colab import drive
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/Group32/ROME_final/pilot_both_v2')
OUT.mkdir(parents=True, exist_ok=True)
source_file = OUT/'source_commit.txt'
if source_file.exists():
    assert source_file.read_text().strip() == SOURCE_COMMIT, "Different source commit. Restore the recorded commit or use a new output folder."
else:
    source_file.write_text(SOURCE_COMMIT + '\n')
GROUPS = ROOT/'results/related_groups/pilot_groups.json'
def stage(n):
    run([PYTHON, ROOT/'scripts/run_multi_stage.py', '--groups', GROUPS,
         '--limit', n, '--order', 'both', '--output', OUT], cwd=ROOT)
    run([PYTHON, ROOT/'scripts/verify_multi_results.py', '--output', OUT, '--limit', n], cwd=ROOT)


Mounted at /content/drive


## One-group smoke test: both orders

In [4]:
stage(1)


data/counterfact.json does not exist. Downloading from https://rome.baulab.info/data/dsets/counterfact.json

  0%|          | 0.00/43.0M [00:00<?, ?B/s]
  0%|          | 128k/43.0M [00:00<03:17, 227kB/s]
  1%|          | 256k/43.0M [00:00<02:01, 371kB/s]
  1%|          | 512k/43.0M [00:00<01:05, 677kB/s]
  2%|▏         | 896k/43.0M [00:01<00:39, 1.12MB/s]
  4%|▍         | 1.75M/43.0M [00:01<00:19, 2.26MB/s]
  8%|▊         | 3.50M/43.0M [00:01<00:09, 4.51MB/s]
 16%|█▌        | 6.88M/43.0M [00:01<00:04, 8.53MB/s]
 23%|██▎       | 9.88M/43.0M [00:01<00:03, 10.8MB/s]
 31%|███       | 13.1M/43.0M [00:02<00:02, 12.7MB/s]
 40%|███▉      | 17.0M/43.0M [00:02<00:01, 15.2MB/s]
 48%|████▊     | 20.8M/43.0M [00:02<00:01, 16.8MB/s]
 57%|█████▋    | 24.4M/43.0M [00:02<00:01, 17.1MB/s]
 65%|██████▌   | 28.1M/43.0M [00:02<00:00, 17.9MB/s]
 74%|███████▍  | 31.9M/43.0M [00:03<00:00, 18.5MB/s]
 83%|████████▎ | 35.6M/43.0M [00:03<00:00, 19.1MB/s]
 92%|█████████▏| 39.5M/43.0M [00:03<00:00, 18.8MB/s]
100%|█

## Five-group pilot: both orders

In [5]:
stage(5)


Loaded dataset with 21919 elements
Resume: verified group_001_forward.json
Resume: verified group_001_reverse.json
Loading pinned GPT-2 XL on GPU...
/content/group32_final_multi/.venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
[1/8 pending] group_002 Helsinki forward: evaluating unedited model
[1/8 pending] group_002 Helsinki forward: edit 1/2: case 1304
[1/8 pending] group_002 Helsinki forward: evaluating all facts after edit 1
[1/8 pending] group_002 Helsinki forward: edit 2/2: case 3182
[1/8 pending] group_002 Helsinki forward: evaluating all facts after edit 2
[1/8 pending] group_002 Helsinki forward: independent control: case 1304
[1/8 pending] group_

## Resume check
This reruns verification only for the five completed groups. Result hashes must remain unchanged.

In [6]:
import hashlib
before = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.glob('group_*.json')}
stage(5)
after = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.glob('group_*.json')}
assert before == after, "Resume changed completed results."
print("RESUME CHECK PASSED")


Loaded dataset with 21919 elements
Resume: verified group_001_forward.json
Resume: verified group_001_reverse.json
Resume: verified group_002_forward.json
Resume: verified group_002_reverse.json
Resume: verified group_003_forward.json
Resume: verified group_003_reverse.json
Resume: verified group_004_forward.json
Resume: verified group_004_reverse.json
Resume: verified group_005_forward.json
Resume: verified group_005_reverse.json
{
  "schema": 2,
  "num_groups": 5,
  "num_group_orders": 10,
  "order": "both",
  "verified": true,
  "orders": {
    "forward": {
      "num_groups": 5,
      "all_edits_hold_count": 5,
      "all_edits_hold_percent": 100.0,
      "first_edit_loss_count": 0,
      "first_edit_eligible_count": 5,
      "first_edit_loss_percent_all_groups": 0.0,
      "first_edit_loss_percent_eligible": 0.0,
      "prior_edits_lost": 0,
      "prior_edits_eligible": 5,
      "prior_edit_loss_percent_eligible": 0.0
    },
    "reverse": {
      "num_groups": 5,
      "all_edit

## Complete 20-group pilot
This resumes the first five groups and completes both orders for all 20. Progress appears at each edit/evaluation stage. Timings depend on the GPU; do not infer completion from elapsed time.

In [7]:
stage(20)


Loaded dataset with 21919 elements
Resume: verified group_001_forward.json
Resume: verified group_001_reverse.json
Resume: verified group_002_forward.json
Resume: verified group_002_reverse.json
Resume: verified group_003_forward.json
Resume: verified group_003_reverse.json
Resume: verified group_004_forward.json
Resume: verified group_004_reverse.json
Resume: verified group_005_forward.json
Resume: verified group_005_reverse.json
Loading pinned GPT-2 XL on GPU...
/content/group32_final_multi/.venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
[1/30 pending] group_006 Windows Vista forward: evaluating unedited model
[1/30 pending] group_006 Windows Vista forw

## Verify and export
The ZIP is created only after re-verification. It contains raw prompt scores, independent controls, summary, CSV, logs, exact source, and environment metadata. The old score-only pilot remains separate.

In [8]:
run([PYTHON, ROOT/'scripts/verify_multi_results.py', '--output', OUT, '--limit', 20], cwd=ROOT)
summary = json.loads((OUT/'summary_20.json').read_text())
assert summary['verified'] and summary['complete_all_groups'] and summary['num_group_orders'] == 40
assert len(list(OUT.glob('group_*.json'))) == 40
archive = shutil.make_archive('/content/Group32_Rohith_Final_Verified_Pilot', 'zip', OUT)
from google.colab import files
files.download(archive)
print("COMPLETE: 20 groups, 40 group/order results; verification passed.")
print("Saved:", archive)


{
  "schema": 2,
  "num_groups": 20,
  "num_group_orders": 40,
  "order": "both",
  "verified": true,
  "orders": {
    "forward": {
      "num_groups": 20,
      "all_edits_hold_count": 15,
      "all_edits_hold_percent": 75.0,
      "first_edit_loss_count": 5,
      "first_edit_eligible_count": 20,
      "first_edit_loss_percent_all_groups": 25.0,
      "first_edit_loss_percent_eligible": 25.0,
      "prior_edits_lost": 5,
      "prior_edits_eligible": 20,
      "prior_edit_loss_percent_eligible": 25.0
    },
    "reverse": {
      "num_groups": 20,
      "all_edits_hold_count": 16,
      "all_edits_hold_percent": 80.0,
      "first_edit_loss_count": 4,
      "first_edit_eligible_count": 20,
      "first_edit_loss_percent_all_groups": 20.0,
      "first_edit_loss_percent_eligible": 20.0,
      "prior_edits_lost": 4,
      "prior_edits_eligible": 20,
      "prior_edit_loss_percent_eligible": 20.0
    }
  },
  "run_key": "b908f54972fd42f2a172183ed512c9d59a83b4dc2f86eaa9a3bae2d211fcee79

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

COMPLETE: 20 groups, 40 group/order results; verification passed.
Saved: /content/Group32_Rohith_Final_Verified_Pilot.zip
